<a href="https://colab.research.google.com/github/1Bur1/Tuwaiq-classes---advanced-AI-python---final-project/blob/main/02_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#task 7
#write a clean_data() function that does all the above steps in order so you can reuse it



def clean_data(df):
  df = df.copy()

  #task1
  print(df.shape)
  #task2
  df.info()

  #task3
  # Converted "Bsmt Full Bath", "Bsmt Half Bath", and "Garage Cars"
  # from float to Int64 because they represent count-based numeric values with possible missing data.
  numeric_fix_cols = [
      "Bsmt Full Bath",
      "Bsmt Half Bath",
      "Garage Cars"
    ]

  for col in numeric_fix_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

  for col in numeric_fix_cols:
    df[col] = df[col].astype("Int64")







# Converted "Year Built", "Year Remod/Add", "Garage Yr Blt", and "Yr Sold"
# from int/float to datetime because they represent time (years) rather than continuous numeric values.
  year_cols = [
      "Year Built",
      "Year Remod/Add",
      "Garage Yr Blt",
      "Yr Sold"
  ]

  for col in year_cols:
      df[col] = pd.to_datetime(df[col], format="%Y", errors="coerce")








# Converted "Order", "PID", and "MS SubClass"
# from int to category because they are identifiers or coded categories, not meaningful numeric features.
  category_fix_cols = [
      "Order",#identifiers
      "PID",#identifiers
      "MS SubClass",#coded
      "Overall Qual",#rate level
      "Overall Cond"#rate level

  ]

  for col in category_fix_cols:
      df[col] = df[col].astype("category")







#because of the type  object it use unneeded amount of memiory
#convert to category
  cat_cols = df.select_dtypes(include='object').columns

  for col in cat_cols:
      df[col] = df[col].astype('category')

  df.info()






#task4
# detect missing
  missing = df.isnull().sum().sort_values(ascending=False)
  print(missing)
  print(missing[missing > 0].head(10))  # columns with the most missing values









# drop the Columns with  high missings
  drop_cols = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']


# Filter drop_cols to only include columns that are actually in df.columns * i put it because of an error
  existing_drop_cols = [col for col in drop_cols if col in df.columns]
  df = df.drop(columns=existing_drop_cols)












# Categorical columns were filled using the mode to preserve the most used category
  cat_cols = df.select_dtypes(include=['object', 'category']).columns

  for col in cat_cols:
      df[col] = df[col].fillna(df[col].mode()[0])







# Numeric columns were filled using the median to handle skewed distributions
  num_cols = df.select_dtypes(include=['int64', 'float64', 'Int64']).columns

  for col in num_cols:
      df[col] = df[col].fillna(df[col].median())


  # fill datetime columns with mode
  date_cols = df.select_dtypes(include=['datetime64[ns]']).columns
  for col in date_cols:
      df[col] = df[col].fillna(df[col].mode()[0])






  #Task 5
  #• Handle duplicates: check with .duplicated().sum() and remove if any

  print(df.duplicated().sum())

  df = df.drop_duplicates() #maybe we will need it in another data




  #task 6
  #Spot outliers: use a boxplot or the IQR method on the target column; cap extreme values at the 99th percentile


  import matplotlib.pyplot as plt





  # Boxplot
  plt.boxplot(df["SalePrice"])
  plt.title("Boxplot of SalePrice")
  plt.show()



  # Cap at 99th percentile
  upper_99 = df["SalePrice"].quantile(0.99)
  df["SalePrice" + "_capped"] = df["SalePrice"].clip(upper=upper_99)




  #task 8
# Add 3 checks at the end (e.g., no nulls in key columns, all target values > 0, correct number of columns)



# No nulls
  assert df["SalePrice"].isnull().sum() == 0

# Target values must be positive
  assert (df["SalePrice"] > 0).all()

# Ensure column exists
  assert "SalePrice" in df.columns

  return df

df = pd.read_csv("AmesHousing.csv")
df_clean = clean_data(df)


In [ ]:
#task1
#• One-hot encode at least 2 categorical columns using pd.get_dummies()


# One-hot encoding for categorical columns
df = pd.get_dummies(df, columns=["Neighborhood", "House Style"], prefix=["Neighborhood", "HouseStyle"])





In [14]:
#task2
#• Ordinal encode 1 ordered column (e.g., quality: low → high)


qual_order = ["Po", "Fa", "TA", "Gd", "Ex"]

df["Kitchen Qual Ord"] = pd.Categorical(df["Kitchen Qual"],categories=qual_order,ordered=True)



print(df["Kitchen Qual Ord"])


#"The column 'Kitchen Qual' was selected because it represents an ordered quality scale (from poor to excellent), making ordinal encoding appropriate to preserve this natural ranking and improve model interpretability."


0       TA
1       TA
2       Gd
3       Ex
4       TA
        ..
2925    TA
2926    TA
2927    TA
2928    TA
2929    TA
Name: Kitchen Qual Ord, Length: 2930, dtype: category
Categories (5, object): ['Po' < 'Fa' < 'TA' < 'Gd' < 'Ex']


In [ ]:
#task3
#• Scale at least 2 numerical columns using StandardScaler


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[["Gr Liv Area", "SalePrice"]] = scaler.fit_transform(
    df[["Gr Liv Area", "SalePrice"]]
)

#"StandardScaler was applied to normalize the numerical features so they have mean 0 and standard deviation 1."

In [21]:
#task4
#Create 2 domain features: a meaningful ratio (e.g., price_per_sqft) and one more of your choice.
#Use safe division to avoid dividing by zero



#price per square foot
df["price_per_sqft"] = df["SalePrice"] / (df["Gr Liv Area"] + 1e-6)

#rooms per area (interaction / ratio)
df["rooms_per_area"] = df["TotRms AbvGrd"] / (df["Gr Liv Area"] + 1e-6)


"""
1. price_per_sqft
"This feature was created to measure property value relative to size, providing a more comparable metric across houses."

2. rooms_per_area
"This feature captures how efficiently space is used, which may reflect layout quality and housing density."

"""


'\n1. price_per_sqft\n"This feature was created to measure property value relative to size, providing a more comparable metric across houses."\n\n2. rooms_per_area\n"This feature captures how efficiently space is used, which may reflect layout quality and housing density."\n\n'

In [ ]:
#task 5
#• Create 1 interaction feature: multiply two columns that logically go together (e.g., quality × area)

# Interaction feature: quality × area
df["qual_area_interaction"] = df["Overall Qual"] * df["Gr Liv Area"]

#"This interaction feature combines quality and size, which may better capture overall property value than each feature individually."


In [ ]:
#task6
#• Log-transform 1 skewed column using np.log1p() — show the histogram before and after




# Before (raw)
plt.hist(df["SalePrice"])
plt.title("SalePrice (raw)")
plt.show()

# Log transform
df["SalePrice_log"] = np.log1p(df["SalePrice"])

# After (log)
plt.hist(df["SalePrice_log"])
plt.title("SalePrice (log1p)")
plt.show()

#"Log transformation was applied to reduce skewness and make the distribution more symmetric."

In [ ]:
#task7
#• Bin 1 column into meaningful groups (e.g., age groups: New, Recent, Old)


# Bin Year Built into meaningful groups
bins = [0, 1950, 2000, 2025]
labels = ["Old", "Mid", "New"]

df["YearBuilt_bin"] = pd.cut(df["Year Built"], bins=bins, labels=labels)

#"Binning was applied to group houses by construction period, making the feature more interpretable and capturing differences between old and new properties."

In [ ]:
#task8
#• Remove redundant features: find highly correlated pairs (r > 0.95) and drop one

# Compute correlation matrix
corr_matrix = df.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find highly correlated features (r > 0.95)
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

print("Columns to drop:", to_drop)

# Drop redundant features
df = df.drop(columns=to_drop)


#"Highly correlated features were removed to reduce redundancy and multicollinearity, which improves model stability and performance."